# S-Gate Training on Kaggle

End-to-end training pipeline adapted for Kaggle:
1. Setup environment
2. Locate datasets (LibriSpeech + MUSAN from `/kaggle/input`)
3. Get project code (clone from GitHub or upload)
4. Wire up datasets with dynamic SNR mixing
5. Auto-tune batch size, AMP, grad clip
6. Train with tqdm + per-epoch metrics
7. Resume from latest checkpoint
8. Validate (SI-SNR / PESQ / STOI)
9. Loss curves + spectrograms + audio
10. Custom audio test

## Kaggle setup checklist
1. **Settings → Accelerator**: choose **GPU T4 x2** (or P100).
2. **Settings → Internet**: turn **ON** (needed for pip / GitHub clone).
3. **Add Data** (right panel) → search and add:
   - `LibriSpeech ASR Corpus` (any train-clean-100 dataset)
   - `MUSAN` (musan corpus)
4. Outputs persist under `/kaggle/working/` (saved with the notebook version).

## 1. Setup environment

In [ ]:
!pip -q install librosa==0.10.2 pesq pystoi soundfile tqdm

import os, sys, time, math, glob, json, subprocess, shutil
import torch
import torchaudio

print('python      :', sys.version.split()[0])
print('torch       :', torch.__version__)
print('torchaudio  :', torchaudio.__version__)
print('cuda avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu         :', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'gpu memory  : {free/1e9:.2f} GB free / {total/1e9:.2f} GB total')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Kaggle paths
INPUT_ROOT  = '/kaggle/input'
WORK_ROOT   = '/kaggle/working'
CKPT_DIR    = f'{WORK_ROOT}/checkpoints'
LOG_DIR     = f'{WORK_ROOT}/logs'
RESULTS_DIR = f'{WORK_ROOT}/results'
PLOTS_DIR   = f'{RESULTS_DIR}/plots'
AUDIO_DIR   = f'{RESULTS_DIR}/audio'
for d in (CKPT_DIR, LOG_DIR, PLOTS_DIR, AUDIO_DIR):
    os.makedirs(d, exist_ok=True)
print('working dir :', WORK_ROOT)
print('inputs      :', sorted(os.listdir(INPUT_ROOT)) if os.path.isdir(INPUT_ROOT) else '(none)')

## 2. Get project code

Two options:
- **(A)** Clone from GitHub (set the repo URL below).
- **(B)** Add this notebook's project as a Kaggle dataset, then copy from `/kaggle/input/<your-dataset>/` to `/kaggle/working/sgate/`.

In [ ]:
# === EDIT THESE ===
GITHUB_REPO_URL = 'https://github.com/vinothsundar-dev/sgate.git'  # public repo or use token URL
USE_GITHUB      = True   # False if you uploaded the project as a Kaggle dataset
PROJECT_DATASET = ''     # e.g. 'username/sgate-project' if USE_GITHUB=False
# ==================

WORK = f'{WORK_ROOT}/sgate'
if os.path.exists(WORK):
    shutil.rmtree(WORK)

if USE_GITHUB:
    print(f'Cloning {GITHUB_REPO_URL} ...')
    res = subprocess.run(['git', 'clone', '-q', GITHUB_REPO_URL, WORK],
                         capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'clone failed: {res.stderr}')
    print('✓ cloned')
else:
    src = f'{INPUT_ROOT}/{PROJECT_DATASET.split("/")[-1] if "/" in PROJECT_DATASET else PROJECT_DATASET}'
    if not os.path.isdir(src):
        raise RuntimeError(f'Project dataset not found at {src}. '
                           f'Add it via Add Data and set PROJECT_DATASET.')
    shutil.copytree(src, WORK)
    print(f'✓ copied from {src}')

os.chdir(WORK)
sys.path.insert(0, WORK)
print('repo files:', sorted(os.listdir('.')))

## 3. Locate datasets

Auto-discovers LibriSpeech (.flac) and MUSAN noise (.wav) under `/kaggle/input/`.
Add the datasets via the **Add Data** panel before running.

In [ ]:
def find_first_with(root, want_substring, ext):
    """Find the first directory under root that contains files of `ext` and whose path includes want_substring."""
    for d, _, files in os.walk(root):
        if want_substring.lower() in d.lower() and any(f.endswith(ext) for f in files):
            # Walk back up to the dataset root containing the substring once.
            return d
    return None

def scan_files(root, ext):
    out = []
    if not root or not os.path.isdir(root):
        return out
    for d, _, files in os.walk(root):
        for fn in files:
            if fn.endswith(ext):
                out.append(os.path.join(d, fn))
    return sorted(out)

# LibriSpeech: look for any directory containing .flac under a librispeech-named folder.
libri_root = None
for entry in os.listdir(INPUT_ROOT):
    if 'libri' in entry.lower():
        candidate = f'{INPUT_ROOT}/{entry}'
        # Use the dataset root, scan_files will find .flac recursively.
        libri_root = candidate
        break

# MUSAN: look for musan/noise/ specifically.
musan_noise_root = None
for entry in os.listdir(INPUT_ROOT):
    if 'musan' in entry.lower():
        candidate = f'{INPUT_ROOT}/{entry}'
        # Try to find a 'noise' subdir; otherwise use the root.
        for d, _, _ in os.walk(candidate):
            if d.endswith('/noise') or '/noise/' in d:
                musan_noise_root = d if d.endswith('/noise') else candidate
                break
        if musan_noise_root is None:
            musan_noise_root = candidate
        break

print('LibriSpeech root:', libri_root)
print('MUSAN noise root:', musan_noise_root)

# Cache file lists to /kaggle/working (faster on subsequent runs).
def cached_scan(root, ext, cache_path, force=False):
    if not force and os.path.exists(cache_path):
        with open(cache_path) as f:
            files = [l.strip() for l in f if l.strip()]
        if files:
            return files
    files = scan_files(root, ext)
    with open(cache_path, 'w') as f:
        f.write('\n'.join(files))
    return files

speech_files = cached_scan(libri_root, '.flac', f'{WORK_ROOT}/speech_files.txt')
noise_files  = cached_scan(musan_noise_root, '.wav', f'{WORK_ROOT}/noise_files.txt')
# MUSAN noise files may live under noise/free-sound and noise/sound-bible; both .wav.
if not noise_files and musan_noise_root:
    # Fallback: scan the whole musan dataset for .wav under any 'noise' folder.
    noise_files = [f for f in scan_files(musan_noise_root, '.wav') if '/noise/' in f or f.endswith('/noise')]
    with open(f'{WORK_ROOT}/noise_files.txt', 'w') as f:
        f.write('\n'.join(noise_files))

print(f'speech files: {len(speech_files)}')
print(f'noise files : {len(noise_files)}')
if len(speech_files) == 0 or len(noise_files) == 0:
    raise RuntimeError('No data found. Add LibriSpeech + MUSAN via the Add Data panel and re-run.')

## 4. Dataset + dynamic SNR mixing

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader

SR = 16000
SEG_SECONDS = 2.0
SEG_SAMPLES = int(SR * SEG_SECONDS)

MAX_SPEECH = 4000
MAX_NOISE  = 1000
if len(speech_files) > MAX_SPEECH:
    speech_files = random.Random(0).sample(speech_files, MAX_SPEECH)
if len(noise_files) > MAX_NOISE:
    noise_files = random.Random(0).sample(noise_files, MAX_NOISE)
print(f'using: {len(speech_files)} speech / {len(noise_files)} noise')

def _load_segment(path, n):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim=0)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    if wav.numel() < n:
        wav = wav.repeat((n // wav.numel()) + 1)
    start = random.randint(0, wav.numel() - n)
    return wav[start:start + n].contiguous()

def _safe_load(files, n, max_retries=4):
    last = None
    for _ in range(max_retries):
        path = random.choice(files)
        try:
            return _load_segment(path, n)
        except Exception as e:
            last = (path, e)
    raise RuntimeError(f'load failed after retries: {last}')

class SpeechNoiseDataset(Dataset):
    def __init__(self, speech, noise, length_per_epoch=2048):
        self.speech = speech
        self.noise  = noise
        self.length = length_per_epoch
    def __len__(self):
        return self.length
    def __getitem__(self, _):
        sp = _safe_load(self.speech, SEG_SAMPLES)
        ns = _safe_load(self.noise,  SEG_SAMPLES)
        sp = sp / sp.abs().max().clamp_min(1e-6) * 0.7
        ns = ns / ns.abs().max().clamp_min(1e-6)
        return sp.float(), ns.float()

def mix_at_snr(clean, noise, snr_db, eps=1e-8):
    cp  = clean.pow(2).mean(dim=-1, keepdim=True) + eps
    np_ = noise.pow(2).mean(dim=-1, keepdim=True).clamp_min(1e-6)
    target_np = cp / (10.0 ** (snr_db.view(-1, 1) / 10.0))
    return clean + noise * (target_np / np_).sqrt()

train_ds = SpeechNoiseDataset(speech_files, noise_files, length_per_epoch=2048)
val_ds   = SpeechNoiseDataset(speech_files[-200:], noise_files[-50:], length_per_epoch=64)
print('dataset ready')

## 5. Model + auto batch size

In [ ]:
from model  import SGateModel, count_parameters
from losses import SGateLoss

model = SGateModel().to(DEVICE)
n_params = count_parameters(model)
print(f'parameters: {n_params:,}')

def find_max_batch(model, max_try=128, start=8):
    if not torch.cuda.is_available():
        return 16
    bs = start; best = start
    model.train()
    while bs <= max_try:
        try:
            torch.cuda.empty_cache()
            x = torch.randn(bs, SEG_SAMPLES, device=DEVICE)
            with torch.amp.autocast(device_type='cuda', enabled=True):
                y = model(x)
                loss = y.float().pow(2).mean()
            loss.backward()
            model.zero_grad(set_to_none=True)
            best = bs; bs *= 2
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                torch.cuda.empty_cache(); break
            raise
    return max(start, best // 2)

BATCH_SIZE = find_max_batch(model, max_try=64, start=8)
print(f'auto batch size: {BATCH_SIZE}')

USE_AMP = torch.cuda.is_available()
GRAD_CLIP = 1.0
NUM_WORKERS = 2  # Kaggle local SSD is fast; workers are stable here.

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS,
)
if NUM_WORKERS > 0:
    loader_kwargs['persistent_workers'] = True
    loader_kwargs['prefetch_factor'] = 2

train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **loader_kwargs)
print(f'loaders ready: batch={BATCH_SIZE} workers={NUM_WORKERS}')

## 6. Training hyperparameters

In [ ]:
EPOCHS         = 30
LR_START       = 1e-5
LR_MAX         = 3e-4
LR_MIN         = 1e-6
WARMUP_STEPS   = 500
PERCEPTUAL_AT  = 6
LOG_EVERY      = 25
SNR_RANGE      = (-5.0, 20.0)

loss_fn = SGateLoss(perceptual=False, sample_rate=SR).to(DEVICE)
optim   = torch.optim.AdamW(model.parameters(), lr=LR_START, weight_decay=1e-4)
scaler  = torch.amp.GradScaler('cuda', enabled=USE_AMP)

steps_per_epoch = len(train_loader)
TOTAL_STEPS     = steps_per_epoch * EPOCHS

history = {'epoch': [], 'total': [], 'sisnr': [], 'mrstft': [],
           'val_sisnr': [], 'val_pesq': [], 'val_stoi': [], 'lr': []}
best_loss = float('inf'); best_ckpt = None

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR_START + (LR_MAX - LR_START) * (step / max(1, WARMUP_STEPS))
    p = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    p = min(1.0, max(0.0, p))
    return LR_MIN + 0.5 * (LR_MAX - LR_MIN) * (1.0 + math.cos(math.pi * p))

def log_line(s):
    print(s)
    with open(f'{LOG_DIR}/train.log', 'a') as f:
        f.write(s + '\n')

## 7. Resume from latest checkpoint (if any)

In [ ]:
def latest_ckpt(d):
    files = sorted(glob.glob(f'{d}/sgate_epoch*.pt'))
    return files[-1] if files else None

start_epoch = 0
global_step = 0
ckpt_path = latest_ckpt(CKPT_DIR)

# Optionally resume from a checkpoint added as a Kaggle dataset.
# Set RESUME_FROM = '/kaggle/input/<your-ckpt-dataset>/sgate_epochNN.pt' to start there.
RESUME_FROM = ''
if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt_path = RESUME_FROM

if ckpt_path:
    ck = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ck['model'])
    if 'optim' in ck:  optim.load_state_dict(ck['optim'])
    if 'scaler' in ck and USE_AMP and ck['scaler'] is not None: scaler.load_state_dict(ck['scaler'])
    start_epoch = ck.get('epoch', 0) + 1
    global_step = ck.get('step', 0)
    if 'history' in ck:
        history = ck['history']
    print(f'resumed from {ckpt_path} -> epoch {start_epoch}, step {global_step}')
else:
    print('no checkpoint, starting fresh')

## 8. Training loop (tqdm + per-epoch metrics)

In [ ]:
from tqdm.auto import tqdm
from losses import si_snr as si_snr_metric

model.train()
skipped = 0
t_run0 = time.time()

for epoch in range(start_epoch, EPOCHS):
    loss_fn.perceptual = epoch >= PERCEPTUAL_AT
    t_ep0 = time.time()
    ep = {'total': 0., 'sisnr': 0., 'mr': 0., 'gnorm': 0., 'sisnr_db': 0., 'n': 0}

    pbar = tqdm(train_loader, desc=f'epoch {epoch:02d}/{EPOCHS}', leave=False, dynamic_ncols=True)
    for clean, noise in pbar:
        clean = clean.to(DEVICE, non_blocking=True)
        noise = noise.to(DEVICE, non_blocking=True)
        snr   = torch.empty(clean.size(0), device=DEVICE).uniform_(*SNR_RANGE)
        noisy = mix_at_snr(clean, noise, snr)
        peak  = noisy.abs().amax(dim=-1, keepdim=True).clamp_min(1e-6)
        gain  = (0.95 / peak).clamp(max=1.0)
        noisy = noisy * gain; clean = clean * gain

        lr = lr_at(global_step)
        for g in optim.param_groups: g['lr'] = lr

        optim.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
            est = model(noisy)
            L = min(est.shape[-1], clean.shape[-1])
            loss, comps = loss_fn(est[..., :L], clean[..., :L])

        if not torch.isfinite(loss):
            skipped += 1; global_step += 1; continue

        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP))
        if not math.isfinite(gnorm):
            skipped += 1; scaler.update(); global_step += 1; continue
        scaler.step(optim); scaler.update()

        with torch.no_grad():
            sisnr_db = float(si_snr_metric(est[..., :L].float(), clean[..., :L].float()).mean())

        ep['total']    += comps['total']
        ep['sisnr']    += comps['sisnr']
        ep['mr']       += comps['mrstft']
        ep['gnorm']    += gnorm
        ep['sisnr_db'] += sisnr_db
        ep['n']        += 1
        pbar.set_postfix(loss=f'{comps["total"]:+.3f}',
                         si_snr=f'{sisnr_db:+.2f}dB',
                         lr=f'{lr:.1e}', skip=skipped)
        global_step += 1

    n = max(ep['n'], 1)
    avg_total = ep['total']/n; avg_sisnr = ep['sisnr']/n; avg_mr = ep['mr']/n
    avg_gnorm = ep['gnorm']/n; avg_sisnr_db = ep['sisnr_db']/n

    history['epoch'].append(epoch)
    history['total'].append(avg_total)
    history['sisnr'].append(avg_sisnr)
    history['mrstft'].append(avg_mr)
    history['lr'].append(lr)

    ck_path = f'{CKPT_DIR}/sgate_epoch{epoch:02d}.pt'
    torch.save({
        'model': model.state_dict(),
        'optim': optim.state_dict(),
        'scaler': scaler.state_dict() if USE_AMP else None,
        'epoch': epoch, 'step': global_step,
        'history': history,
    }, ck_path)
    if avg_total < best_loss:
        best_loss = avg_total; best_ckpt = ck_path

    log_line(
        f'epoch {epoch:02d} | loss={avg_total:+.4f} sisnr={avg_sisnr_db:+.2f}dB '
        f'mr={avg_mr:.4f} |g|={avg_gnorm:.3f} lr={lr:.2e} '
        f'time={time.time()-t_ep0:.1f}s skipped={skipped} -> {ck_path}'
    )
    with open(f'{LOG_DIR}/history.json', 'w') as f:
        json.dump(history, f, indent=2)

## 9. Validation: SI-SNR + PESQ + STOI

In [ ]:
import numpy as np
from losses import si_snr
try:
    from pesq import pesq as pesq_fn
    from pystoi import stoi as stoi_fn
    HAS_METRICS = True
except Exception as e:
    print('metrics unavailable:', e)
    HAS_METRICS = False

@torch.no_grad()
def validate(model, loader, max_batches=8):
    model.eval()
    sisnrs, pesqs, stois = [], [], []
    for i, (clean, noise) in enumerate(loader):
        if i >= max_batches: break
        clean = clean.to(DEVICE); noise = noise.to(DEVICE)
        snr = torch.full((clean.size(0),), 5.0, device=DEVICE)
        noisy = mix_at_snr(clean, noise, snr)
        est = model(noisy)
        L = min(est.shape[-1], clean.shape[-1])
        sisnrs.append(si_snr(est[..., :L], clean[..., :L]).mean().item())
        if HAS_METRICS:
            for j in range(clean.size(0)):
                c = clean[j, :L].cpu().numpy().astype(np.float32)
                e = est[j,  :L].cpu().numpy().astype(np.float32)
                try: pesqs.append(pesq_fn(SR, c, e, 'wb'))
                except Exception: pass
                try: stois.append(stoi_fn(c, e, SR, extended=False))
                except Exception: pass
    model.train()
    out = {'si_snr_dB': float(np.mean(sisnrs)) if sisnrs else float('nan')}
    if pesqs: out['pesq'] = float(np.mean(pesqs))
    if stois: out['stoi'] = float(np.mean(stois))
    return out

metrics = validate(model, val_loader)
print('validation:', metrics)
with open(f'{LOG_DIR}/val_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

## 10. Loss curves

In [ ]:
import matplotlib.pyplot as plt

hist_path = f'{LOG_DIR}/history.json'
if os.path.exists(hist_path):
    with open(hist_path) as f: history = json.load(f)

if not history.get('epoch'):
    print('No history yet.')
else:
    ep = history['epoch']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(ep, history['total'],  marker='o'); axes[0].set_title('total loss')
    axes[1].plot(ep, history['sisnr'],  marker='o', color='C1'); axes[1].set_title('SI-SNR loss')
    axes[2].plot(ep, history['mrstft'], marker='o', color='C2'); axes[2].set_title('MR-STFT loss')
    for a in axes: a.set_xlabel('epoch'); a.grid(alpha=0.3)
    plt.tight_layout()
    out = f'{PLOTS_DIR}/loss_curves.png'
    plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
    print('saved', out)

## 11. Spectrograms + audio playback

In [ ]:
import numpy as np, librosa, librosa.display
from IPython.display import Audio, display
import soundfile as sf

def _spec_db(wav, n_fft=512, hop=128):
    S = np.abs(librosa.stft(wav, n_fft=n_fft, hop_length=hop)) + 1e-7
    return librosa.amplitude_to_db(S, ref=np.max)

def show_compare(noisy, enhanced, clean=None, sr=SR, title='', save=None):
    cols = 3 if clean is not None else 2
    fig, axes = plt.subplots(1, cols, figsize=(5*cols, 4), sharey=True)
    items = [('noisy', noisy), ('enhanced', enhanced)]
    if clean is not None: items.append(('clean', clean))
    for ax, (name, w) in zip(axes, items):
        librosa.display.specshow(_spec_db(w), sr=sr, hop_length=128,
                                 x_axis='time', y_axis='hz', ax=ax, cmap='magma')
        ax.set_title(name); ax.set_ylim(0, sr/2)
    if title: fig.suptitle(title)
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=120, bbox_inches='tight'); print('saved', save)
    plt.show()
    print('noisy:');    display(Audio(noisy,    rate=sr))
    print('enhanced:'); display(Audio(enhanced, rate=sr))
    if clean is not None:
        print('clean:');    display(Audio(clean, rate=sr))

infer_model = SGateModel().to(DEVICE).eval()
ck = latest_ckpt(CKPT_DIR)
if ck:
    infer_model.load_state_dict(torch.load(ck, map_location=DEVICE)['model'])
    print('loaded', ck)

clean_t, noise_t = val_ds[0]
clean_b = clean_t.unsqueeze(0).to(DEVICE)
noise_b = noise_t.unsqueeze(0).to(DEVICE)
noisy_b = mix_at_snr(clean_b, noise_b, torch.tensor([5.0], device=DEVICE))
with torch.no_grad():
    enhanced_b = infer_model(noisy_b)

sf.write(f'{AUDIO_DIR}/val_clean.wav',    clean_b.cpu().numpy()[0], SR)
sf.write(f'{AUDIO_DIR}/val_noisy.wav',    noisy_b.cpu().numpy()[0], SR)
sf.write(f'{AUDIO_DIR}/val_enhanced.wav', enhanced_b.cpu().numpy()[0], SR)
show_compare(noisy_b.cpu().numpy()[0],
             enhanced_b.cpu().numpy()[0],
             clean_b.cpu().numpy()[0],
             title='val sample @ 5dB SNR',
             save=f'{PLOTS_DIR}/specs_val_sample.png')

## 12. Test on your own audio

Add an audio file as a Kaggle dataset (e.g. `/kaggle/input/my-audio/clip.wav`)
and set `INPUT_AUDIO` below.

In [ ]:
INPUT_AUDIO = ''  # e.g. '/kaggle/input/my-audio/clip.wav'

if not INPUT_AUDIO or not os.path.exists(INPUT_AUDIO):
    print('Set INPUT_AUDIO to a wav/mp3/flac path under /kaggle/input/.')
else:
    wav, sr = torchaudio.load(INPUT_AUDIO)
    wav = wav.mean(dim=0)
    if sr != SR:
        wav = torchaudio.functional.resample(wav, sr, SR)
    wav = wav / wav.abs().max().clamp_min(1e-6) * 0.95
    noisy_b = wav.unsqueeze(0).to(DEVICE)

    infer_model = SGateModel().to(DEVICE).eval()
    ck = latest_ckpt(CKPT_DIR)
    if ck:
        infer_model.load_state_dict(torch.load(ck, map_location=DEVICE)['model'])
    with torch.no_grad():
        enhanced_b = infer_model(noisy_b)

    base = os.path.splitext(os.path.basename(INPUT_AUDIO))[0]
    out_n = f'{AUDIO_DIR}/{base}_noisy.wav'
    out_e = f'{AUDIO_DIR}/{base}_enhanced.wav'
    sf.write(out_n, noisy_b.cpu().numpy()[0], SR)
    sf.write(out_e, enhanced_b.cpu().numpy()[0], SR)
    print('saved:', out_n)
    print('saved:', out_e)
    show_compare(noisy_b.cpu().numpy()[0],
                 enhanced_b.cpu().numpy()[0],
                 title=f'custom: {base}',
                 save=f'{PLOTS_DIR}/specs_{base}.png')

## 13. Checkpoint info

In [ ]:
ckpts = sorted(glob.glob(f'{CKPT_DIR}/sgate_epoch*.pt'))
print(f'found {len(ckpts)} checkpoints in {CKPT_DIR}')
best = (None, float('inf'))
for p in ckpts:
    ck = torch.load(p, map_location='cpu')
    ep = ck.get('epoch', '?')
    h  = ck.get('history', {}).get('total', [])
    last = h[-1] if h else float('nan')
    print(f'  ep {ep:>3} loss={last:+.4f}  size={os.path.getsize(p)/1e6:.1f}MB  {p}')
    if h and h[-1] < best[1]:
        best = (p, h[-1])
print()
print(f'latest: {ckpts[-1] if ckpts else "(none)"}')
print(f'best:   {best[0]} (loss={best[1]:+.4f})')